# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset version: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Review available RecordSets, fields, and their IDs. Each entity in Croissant can be referenced by its `@id` for unambiguous referencing.

**RecordSets:**

In [ ]:
# List available RecordSets and display their IDs and fields.
record_sets = []
for obj in metadata.iter_objects():
    if getattr(obj, '@type', None) == 'RecordSet' or getattr(obj, '@type', None) == 'cr:RecordSet':
        record_sets.append(obj)

if not record_sets:
    print("No RecordSets found in the top-level schema. Attempting to scan all objects...")
    # Fallback scan: Croissant sometimes defines recordSet as a property
    if hasattr(metadata, 'recordSet'):
        if isinstance(metadata.recordSet, (list, tuple)):
            for rec in metadata.recordSet:
                record_sets.append(rec)
        else:
            record_sets.append(metadata.recordSet)

print(f"Found {len(record_sets)} RecordSet(s):\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. Name: {getattr(rs, 'name', 'N/A')}  |  @id: {getattr(rs, '@id', 'N/A')}")
    fields = getattr(rs, 'field', [])
    if fields:
        print("   Fields (with @id):")
        for f in (fields if isinstance(fields, (list,tuple)) else [fields]):
            print(f"     - {getattr(f, 'name', 'N/A')} (@id: {getattr(f, '@id', 'N/A')})")
    print("")

## 3. Data Extraction
Load records from the available RecordSets using their `@id`. All entities (record sets, fields, columns) are referenced by their `@id` for consistency.

We'll display all discovered RecordSet IDs, then extract records from each into a pandas DataFrame.

In [ ]:
# Gather all RecordSet @id values
record_set_ids = []
for rs in record_sets:
    rid = getattr(rs, '@id', None)
    if rid:
        record_set_ids.append(rid)

# Extract all records from each RecordSet into a dict of DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f'Loaded {len(df)} records from record_set: {record_set_id}')
        dataframes[record_set_id] = df
    except Exception as ex:
        print(f"Could not load records from {record_set_id}: {ex}")

# For demonstration, display column names and head for the first found RecordSet
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns for RecordSet {first_rs_id}: ", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Here, we apply sample data processing and cleaning on a numeric field from a RecordSet. All field and column references use `@id`.

**Example:** Filtering and normalizing the `phq9_total_score` field (PHQ-9 total), which is a common numeric outcome in mental health datasets.

In [ ]:
# Adapt to your dataset: set the appropriate RecordSet @id and numeric field @id
# Use the overview cells above to find the field @id for the PHQ-9 total score field

# For demonstration, we'll guess likely field and group field IDs based on usual patterns.
# Replace these with the actual @id values you found in the overview.
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# List numeric-like fields in this DataFrame
potential_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
print("Potential numeric fields:", potential_numeric_fields)

# Attempt to use common PHQ-9/GAD-7/PSQ fields
preferred_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ["phq9", "gad7", "psq"]) and ('total' in col.lower() or 'score' in col.lower())]
print("Preferred numeric fields:", preferred_numeric_fields)
numeric_field = preferred_numeric_fields[0] if preferred_numeric_fields else (potential_numeric_fields[0] if potential_numeric_fields else None)
print(f"Will use numeric field: {numeric_field}")

if numeric_field:
    # Example threshold for PHQ-9: >10 indicates moderate to severe symptoms
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} found.")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a demographic field, e.g., 'gender' or similar
    group_field = None
    for candidate in df.columns:
        if any(k in candidate.lower() for k in ["gender", "sex", "age", "village", "education"]):
            group_field = candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df)
    else:
        print("No demographic group field (e.g., 'gender') found for grouping.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field (e.g., PHQ-9 total score), and, if available, separation by a grouping field (e.g., gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable field for visualization. Please check column names and types.")

## 6. Conclusion

- We used `mlcroissant` to load and explore the FAIR² Mental Health Survey dataset defined with a Croissant schema.
- Record sets, fields, and columns were referenced programmatically, always using their `@id`, enabling robust analytics and code re-use for FAIR datasets.
- Using exploratory data analysis and visualization methods, we demonstrated how to filter, normalize, and compare mental health scale summary scores by demographic fields, providing actionable insights from open, AI-ready health data.

Continue further by applying more advanced analyses, machine learning, or linking multiple record sets!